# Exploratory Data Analysis & Assumption Verification

This notebook verifies the core statistical assumptions for multiple linear regression:
1. **Linearity**: Visual check of independent variables against Price.
2. **Multicollinearity**: Variance Inflation Factor (VIF) check.
3. **Homoscedasticity**: Analysis of residuals vs fitted values.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Append src path to load data_prep module
sys.path.append(os.path.abspath('../src'))
from data_prep import load_and_validate_data

## Load and Clean Data

We'll load the raw data through the validation and preparation pipeline (outlier removal via IQR).

In [ ]:
data_path = '../data/housing.csv'

# If data doesn't exist, let's generate it
if not os.path.exists(data_path):
    sys.path.append(os.path.abspath('../src'))
    from generate_data import generate_synthetic_housing_data
    generate_synthetic_housing_data(data_path)

df = load_and_validate_data(data_path)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## 1. Linearity Verification

We plot independent features (`SquareFeet`, `Bedrooms`, `Bathrooms`, `AgeOfHouse`) vs. the target variable `Price` along with a linear regression fit line.

In [ ]:
features = ['SquareFeet', 'Bedrooms', 'Bathrooms', 'AgeOfHouse']
target = 'Price'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.scatterplot(data=df, x=col, y=target, ax=axes[i], alpha=0.6)
    sns.regplot(data=df, x=col, y=target, ax=axes[i], scatter=False, color='red')
    axes[i].set_title(f'{col} vs Price')

plt.tight_layout()
plt.show()

## 2. Multicollinearity (VIF Check)

Highly correlated independent features distort coefficients and increase their variance. We calculate the Variance Inflation Factor (VIF):
- VIF = 1: No correlation.
- VIF between 1 and 5: Moderate correlation.
- VIF > 5 (or 10): High multicollinearity.

In [ ]:
X = df[features].copy()
X_with_const = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_with_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]

vif_data

## 3. Homoscedasticity Verification

We fit an ordinary least squares (OLS) regression model and plot the residuals against the fitted values. If residuals are randomly dispersed around the horizontal line $y=0$ with constant vertical spread, homoscedasticity is satisfied.

In [ ]:
y = df[target]
model_ols = sm.OLS(y, X_with_const).fit()

fitted_values = model_ols.fittedvalues
residuals = model_ols.resid

plt.figure(figsize=(8, 5))
sns.scatterplot(x=fitted_values, y=residuals, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values (Homoscedasticity Check)')
plt.show()